<a href="https://colab.research.google.com/github/VasilisPapageorgiou/Statistical-Embeddings/blob/main/StatEmbedRealData_Version2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# -*- coding: utf-8 -*-
"""Revised UCI four-hospital Heart Disease study.

Run as ONE Colab cell or as a Python file. The outputs and model settings of
StatEmbedRealData(1).ipynb are retained for both repeated stratified CV and LOSO.

Changes:
  1. Remove full-row duplicates BEFORE any split (920 -> 918 for supplied data).
  2. Exclude 'chol' from ALL predictors and descriptive statistics.
  3. Convert missing-value markers to NaN and impossible trestbps=0 to NaN.
  4. Fit median imputer ONLY on the internal training subset of EACH outer
     CV/LOSO split; transform validation and test without leakage.
  5. Fit StandardScaler on the same internal training subset after imputation.
  6. Preserve all original CSV tables and PNG plot names across both analyses.
  7. Save Google Drive checkpoints every 10 minutes (at completed epoch boundaries)
     and after every finished CV/LOSO model run. Resume or restart on launch.

Note: The supplied 920-record CSV was previously preprocessed and includes
fractional values for some categorical fields. A median imputer cannot undo
imputation that may already have occurred upstream. Use source raw data and
rebuild if you need a strictly raw-to-result reproducible preprocessing audit.
"""
import os
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

# Configure at the top of the script if needed. Defaults preserve notebook paths.
DATA_PATH_OVERRIDE = os.environ.get("HEART_DATA_PATH", "heart_disease.csv")

# In Colab, mount Drive BEFORE starting any experiment. All tables, figures,
# dataset audit, and checkpoints are stored in this persistent folder.
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DEFAULT_RESULTS_PARENT = '/content/drive/MyDrive/Heart_Disease_918_Source_Embeddings'
else:
    DEFAULT_RESULTS_PARENT = './Heart_Disease_918_Source_Embeddings'
RESULTS_PARENT = os.path.abspath(os.environ.get('HEART_RESULTS_PARENT', DEFAULT_RESULTS_PARENT))
os.makedirs(RESULTS_PARENT, exist_ok=True)
PERSISTENT_INPUT_DIR = os.path.join(RESULTS_PARENT, 'input')
os.makedirs(PERSISTENT_INPUT_DIR, exist_ok=True)
PERSISTENT_INPUT_CSV = os.path.join(PERSISTENT_INPUT_DIR, 'heart_disease_original_input.csv')



# ============================================================
# PERSISTENT, ATOMIC 10-MINUTE CHECKPOINTS / RESUME / RESTART
# ============================================================
import copy as _ck_copy
import datetime as _ck_dt
import hashlib as _ck_hashlib
import json as _ck_json
import random as _ck_random
import time as _ck_time

_CHECKPOINT_VERSION = 'heart-918-no-chol-median-cv-loso-v2'
_CHECKPOINT_INTERVAL_SECONDS = float(os.environ.get('HEART_CHECKPOINT_SECONDS', '600'))
CHECKPOINT_DIR = os.path.join(RESULTS_PARENT, 'checkpoints')
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR, 'heart_study_checkpoint.pt')

class _HeartCheckpoints:
    def __init__(self, filename):
        self.filename = filename
        self.state = {'version': _CHECKPOINT_VERSION, 'signatures': {}, 'completed': {},
                      'partial': {}, 'created': _ck_dt.datetime.now().isoformat()}
        self.current_unit = None
        self._last_save = _ck_time.monotonic()
        if os.path.exists(self.filename):
            choice = os.environ.get('HEART_CHECKPOINT_ACTION', '').strip().lower()
            if choice not in {'resume', 'restart'}:
                while True:
                    try:
                        choice = input('\nExisting Google Drive checkpoint found. [R]esume or [S]tart from scratch? [R]: ').strip().lower()
                    except EOFError:
                        choice = 'r'
                    if choice in {'', 'r', 'resume'}:
                        choice = 'resume'; break
                    if choice in {'s', 'restart', 'start'}:
                        choice = 'restart'; break
                    print('Please type R (resume) or S (restart).')
            if choice == 'resume':
                import torch
                loaded = torch.load(self.filename, map_location='cpu', weights_only=False)
                if loaded.get('version') != _CHECKPOINT_VERSION:
                    raise ValueError('Checkpoint version mismatch; choose Restart to start a new run.')
                self.state = loaded
                print('[CHECKPOINT] RESUME:', len(self.state['completed']), 'completed model runs.',
                      len(self.state['partial']), 'partially trained models.')
            else:
                suffix = _ck_dt.datetime.now().strftime('%Y%m%d_%H%M%S_%f')
                archived = os.path.join(CHECKPOINT_DIR, f'heart_study_checkpoint_backup_{suffix}.pt')
                os.replace(self.filename, archived)
                print('[CHECKPOINT] RESTART: previous checkpoint archived at', archived)
                self.save(force=True)
        else:
            print('[CHECKPOINT] No existing checkpoint. Starting a new run.')
            self.save(force=True)
        print('[CHECKPOINT] Saved to:', self.filename)
        print('[CHECKPOINT] Interval:', _CHECKPOINT_INTERVAL_SECONDS, 'seconds, plus each completed model run.')

    def register(self, phase, cfg, df):
        # Reject silently mixing results from different data or hyperparameters.
        skip = {'data_path', 'output_dir', 'device', 'num_workers'}
        settings = {k: v for k, v in vars(cfg).items() if k not in skip}
        settings = _ck_json.loads(_ck_json.dumps(settings, default=str))
        digest = _ck_hashlib.sha256(pd.util.hash_pandas_object(df, index=True).values.tobytes()).hexdigest()
        signature = _ck_hashlib.sha256(_ck_json.dumps({'settings': settings,
                         'data_digest': digest, 'features': list(df.columns),
                         'version': _CHECKPOINT_VERSION}, sort_keys=True).encode()).hexdigest()
        existing = self.state['signatures'].get(phase)
        if existing and existing != signature:
            raise ValueError(f'Existing checkpoint for {phase} used different data or model settings. '
                             'Restart to run with the new configuration (old checkpoint is archived).')
        if not existing:
            self.state['signatures'][phase] = signature
            self.save(force=True)
        print(f'[CHECKPOINT] {phase.upper()} configuration verified; input fingerprint {digest[:12]}.')

    def get_completed(self, unit):
        return self.state['completed'].get(unit)

    def get_partial(self, unit):
        return self.state['partial'].get(unit)

    def _rng_state(self):
        import torch
        state = {'python': _ck_random.getstate(), 'numpy': np.random.get_state(),
                 'torch_cpu': torch.get_rng_state()}
        if torch.cuda.is_available():
            state['torch_cuda'] = torch.cuda.get_rng_state_all()
        return state

    def _restore_rng(self, state):
        import torch
        _ck_random.setstate(state['python'])
        np.random.set_state(state['numpy'])
        torch.set_rng_state(state['torch_cpu'])
        if torch.cuda.is_available() and 'torch_cuda' in state:
            torch.cuda.set_rng_state_all(state['torch_cuda'])

    def restore_training(self, unit, model, optimizer, scheduler):
        part = self.get_partial(unit)
        if not part:
            return 0, None, float('inf'), 0, []
        model.load_state_dict(part['model_state'])
        optimizer.load_state_dict(part['optimizer_state'])
        scheduler.load_state_dict(part['scheduler_state'])
        # Must restore *after* constructors have consumed random numbers.
        self._restore_rng(part['rng_state'])
        epoch = part['epoch']
        print(f'[CHECKPOINT] Resuming {unit} at epoch {epoch+1}.')
        return epoch, part['best_model_state'], part['best_valid_loss'], part['epochs_without_improvement'], part['history']

    def save_epoch_if_due(self, unit, epoch, model, optimizer, scheduler,
                          best_model_state, best_valid_loss, epochs_without_improvement, history):
        if _ck_time.monotonic() - self._last_save < _CHECKPOINT_INTERVAL_SECONDS:
            return
        # Epoch-level resume: optimizer, scheduler, early stopping and RNG are
        # restored exactly at the last completed epoch boundary.
        self.state['partial'][unit] = {
            'epoch': epoch+1, 'model_state': _ck_copy.deepcopy(model.state_dict()),
            'optimizer_state': _ck_copy.deepcopy(optimizer.state_dict()),
            'scheduler_state': _ck_copy.deepcopy(scheduler.state_dict()),
            'best_model_state': _ck_copy.deepcopy(best_model_state),
            'best_valid_loss': best_valid_loss,
            'epochs_without_improvement': epochs_without_improvement,
            'history': list(history), 'rng_state': self._rng_state(),
        }
        self.save(force=True)
        print(f'[CHECKPOINT] 10-minute save: {unit}, epoch {epoch+1}.')

    def complete(self, unit, **result):
        self.state['completed'][unit] = result
        self.state['partial'].pop(unit, None)
        self.save(force=True)
        print(f'[CHECKPOINT] Completed {unit}; {len(self.state["completed"])} runs stored.')

    def save(self, force=False):
        if not force and _ck_time.monotonic() - self._last_save < _CHECKPOINT_INTERVAL_SECONDS:
            return
        import torch
        temp = self.filename + '.tmp'
        try:
            torch.save(self.state, temp)
            os.replace(temp, self.filename)
        finally:
            if os.path.exists(temp):
                os.remove(temp)
        self._last_save = _ck_time.monotonic()

_CKPT = _HeartCheckpoints(CHECKPOINT_FILE)

def resolve_heart_data_path(path):
    """Find provided CSV; in Colab permit direct upload if not already present."""
    names = [path, DATA_PATH_OVERRIDE, "heart_disease.csv",
             "heart_disease_918_deduplicated.csv"]
    paths = []
    for name in dict.fromkeys(names):
        f = Path(name).expanduser()
        paths.extend([f, Path("/content") / f.name,
                      Path("/content/drive/MyDrive") / f.name,
                      Path("/mnt/data") / f.name])
    for p in paths:
        if p.is_file():
            if os.path.abspath(str(p)) != os.path.abspath(PERSISTENT_INPUT_CSV):
                if not os.path.isfile(PERSISTENT_INPUT_CSV):
                    import shutil
                    shutil.copy2(p, PERSISTENT_INPUT_CSV)
                    print('[DATA] Original input backed up to:', PERSISTENT_INPUT_CSV)
            return str(p)
    if os.path.isfile(PERSISTENT_INPUT_CSV):
        print('[DATA] Reusing original CSV from persistent backup.')
        return PERSISTENT_INPUT_CSV
    try:
        from google.colab import files
        print("CSV not found. Upload heart_disease.csv (920 records or 918 cleaned records).")
        uploaded = files.upload()
        csvs = [name for name in uploaded if name.lower().endswith(".csv")]
        if csvs:
            import shutil
            shutil.copy2(csvs[0], PERSISTENT_INPUT_CSV)
            print('[DATA] Original input backed up to:', PERSISTENT_INPUT_CSV)
            return csvs[0]
    except ImportError:
        pass
    raise FileNotFoundError("Could not find heart_disease.csv; set HEART_DATA_PATH or upload it in Colab.")


def load_clean_heart_dataframe(path, audit=True):
    """Cleaning is deterministic, and happens before train/validation/test splits."""
    actual = resolve_heart_data_path(path)
    df = pd.read_csv(actual, na_values=["?", "NA", "N/A", "null", "None", ""])
    required = {"source", "target"}
    if not required.issubset(df.columns):
        raise ValueError(f"Missing required columns: {required - set(df.columns)}")
    if df[["source", "target"]].isna().any().any():
        raise ValueError("Target or source labels contain missing values; cannot safely impute those.")
    original_n = len(df)
    # Include target and source in duplicate key. Remove exactly repeated records.
    n_duplicates = int(df.duplicated(keep="first").sum())
    df = df.drop_duplicates(keep="first").reset_index(drop=True)
    if "chol" in df.columns:
        df = df.drop(columns=["chol"])
    if "cholesterol" in df.columns:
        df = df.drop(columns=["cholesterol"])
    feat = [c for c in df.columns if c not in ("source", "target")]
    for col in feat:
        df[col] = pd.to_numeric(df[col], errors="coerce")
    # A zero mmHg resting blood pressure is physiologically impossible;
    # treat it as missing, rather than using it as a genuine measurement.
    if "trestbps" in df:
        df.loc[df["trestbps"] <= 0, "trestbps"] = np.nan
    df[feat] = df[feat].replace([np.inf, -np.inf], np.nan)
    if not set(df["target"].dropna().unique()).issubset({0, 1}):
        raise ValueError("Expected binary target coded 0/1. Check the input data first.")
    # Keep original name strings: they are genuine UCI site labels.
    if audit:
        print(f"[DATA] {actual}: original={original_n}, duplicates_removed={n_duplicates}, "
              f"retained={len(df)}, predictors={len(feat)} (cholesterol excluded)")
        print("[DATA] Hospitals:", df["source"].value_counts().sort_index().to_dict())
        missing = df[feat].isna().sum()
        print("[DATA] Missing values BEFORE fold-wise imputation:", missing[missing > 0].to_dict())
        if original_n == 920 and len(df) != 918:
            raise AssertionError("Expected 918 after removing two duplicates from the supplied 920-row CSV.")
        # This audit copy intentionally retains NaNs: folds must impute independently.
        audit_dir = os.path.join(RESULTS_PARENT, "heart_disease_preprocessing_audit")
        os.makedirs(audit_dir, exist_ok=True)
        df.to_csv(os.path.join(audit_dir, "heart_disease_918_no_chol_before_imputation.csv"), index=False)
        source_table = df.groupby("source")["target"].agg(n="size", positives="sum", prevalence="mean").reset_index()
        source_table.to_csv(os.path.join(audit_dir, "source_counts_prevalence.csv"), index=False)
    return df


def foldwise_median_impute_and_scale(X_train, X_valid, X_test):
    """Fit preprocessing only on INTERNAL training data; no validation/test leakage."""
    # keep_empty_features guards against all-missing columns in a training fold.
    try:
        imp = SimpleImputer(strategy="median", keep_empty_features=True)
    except TypeError:
        imp = SimpleImputer(strategy="median")
    xt = imp.fit_transform(X_train)
    xv = imp.transform(X_valid)
    xe = imp.transform(X_test)
    if xt.shape[1] != X_train.shape[1]:
        raise ValueError("A predictor is entirely missing in this training split. "
                         "Use scikit-learn >=1.2 (keep_empty_features=True) or remove it consistently.")
    scaler = StandardScaler()
    xt = scaler.fit_transform(xt).astype(np.float32)
    xv = scaler.transform(xv).astype(np.float32)
    xe = scaler.transform(xe).astype(np.float32)
    if not (np.isfinite(xt).all() and np.isfinite(xv).all() and np.isfinite(xe).all()):
        raise ValueError("Imputation/scaling left non-finite predictors.")
    return xt, xv, xe



# ============= STUDY 1: STRATIFIED CV + ALL EXISTING FIGURES =============
"""
Complete application pipeline for the uploaded heart_disease.csv dataset.

This version explores multiple source-embedding dimensions:
    m in {2, 4, 8, 16, 32, 64, 128}

Models compared
---------------
1. source_agnostic_mlp
2. source_aware_embedding_mlp_m{m}   for each embedding dimension m
3. source_aware_onehot_mlp

Evaluation
----------
- repeated stratified cross-validation
- internal stratified validation split
- early stopping
- AdamW
- ReduceLROnPlateau
- gradient clipping
- batch normalization
- dropout
- class-weighted loss

Outputs
-------
Tables:
    cv_fold_results.csv
    cv_summary.csv
    sourcewise_fold_results.csv
    sourcewise_summary.csv
    oof_predictions.csv
    training_histories.csv

Figures:
    figure_auc_cv.png
    figure_balanced_accuracy_cv.png
    figure_f1_cv.png
    figure_logloss_cv.png
    figure_roc_oof.png
    figure_calibration_oof.png
    figure_embedding_dimension_auc.png
    figure_embedding_dimension_bacc.png
    figure_embedding_dimension_logloss.png
    figure_sourcewise_auc_best_embedding_vs_agnostic.png
    figure_sourcewise_bacc_best_embedding_vs_agnostic.png
    figure_training_curve_best_embedding.png
"""

import os
import copy
import random
from dataclasses import dataclass
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedStratifiedKFold, train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    roc_curve,
    brier_score_loss,
)
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# configuration
# ============================================================

@dataclass
class Config:
    data_path: str = DATA_PATH_OVERRIDE
    output_dir: str = os.path.join(RESULTS_PARENT, "heart_disease_application_embedding_sweep")

    random_seed: int = 5

    n_splits: int = 10
    n_repeats: int = 1
    valid_size: float = 0.3

    batch_size: int = 64
    max_epochs: int = 200
    patience: int = 10

    learning_rate: float = 1e-2
    weight_decay: float = 1e-4
    min_lr: float = 1e-6
    dropout: float = 0.25
    hidden_dims: Tuple[int, ...] = (32,16)

    embedding_dims: Tuple[int, ...] = (4, 8, 16, 32, 64, 128)

    num_workers: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)

TABLE_DIR = os.path.join(CFG.output_dir, "tables")
FIG_DIR = os.path.join(CFG.output_dir, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


# ============================================================
# reproducibility
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.random_seed)


# ============================================================
# dataset utilities
# ============================================================

class TabularDataset(Dataset):
    def __init__(
        self,
        x_num: np.ndarray,
        source_idx: np.ndarray,
        source_onehot: np.ndarray,
        y: np.ndarray,
    ):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.source_idx = torch.tensor(source_idx, dtype=torch.long)
        self.source_onehot = torch.tensor(source_onehot, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return (
            self.x_num[idx],
            self.source_idx[idx],
            self.source_onehot[idx],
            self.y[idx],
        )


def load_and_prepare_data(path: str):
    df = load_clean_heart_dataframe(path)

    if "target" not in df.columns:
        raise ValueError("The dataset must contain a 'target' column.")
    if "source" not in df.columns:
        raise ValueError("The dataset must contain a 'source' column.")

    feature_cols = [c for c in df.columns if c not in ["target", "source"]]
    X = df[feature_cols].astype(float).values
    y = df["target"].astype(int).values
    source_raw = df["source"].astype(str).values

    source_encoder = LabelEncoder()
    source_idx = source_encoder.fit_transform(source_raw)
    n_sources = len(source_encoder.classes_)

    source_onehot = np.zeros((len(source_idx), n_sources), dtype=float)
    source_onehot[np.arange(len(source_idx)), source_idx] = 1.0

    return df, X, y, source_idx, source_onehot, feature_cols, source_encoder


def make_loaders(
    x_train: np.ndarray,
    s_train: np.ndarray,
    s1_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    s_valid: np.ndarray,
    s1_valid: np.ndarray,
    y_valid: np.ndarray,
):
    train_ds = TabularDataset(x_train, s_train, s1_train, y_train)
    valid_ds = TabularDataset(x_valid, s_valid, s1_valid, y_valid)

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        drop_last=(len(train_ds) % CFG.batch_size == 1),
        pin_memory=torch.cuda.is_available(),
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, valid_loader


# ============================================================
# models
# ============================================================

class SourceAgnosticMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Tuple[int, ...], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        self.backbone = nn.Sequential(*layers)
        self.output = nn.Linear(prev_dim, 1)

    def forward(self, x_num: torch.Tensor, source_idx=None, source_onehot=None):
        h = self.backbone(x_num)
        return self.output(h).squeeze(1)


class SourceAwareEmbeddingMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_sources: int,
        source_embedding_dim: int,
        hidden_dims: Tuple[int, ...],
        dropout: float,
    ):
        super().__init__()
        self.source_embedding = nn.Embedding(n_sources, source_embedding_dim)

        layers = []
        prev_dim = input_dim + source_embedding_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        self.backbone = nn.Sequential(*layers)
        self.output = nn.Linear(prev_dim, 1)

    def forward(self, x_num: torch.Tensor, source_idx: torch.Tensor, source_onehot=None):
        e = self.source_embedding(source_idx)
        x = torch.cat([x_num, e], dim=1)
        h = self.backbone(x)
        return self.output(h).squeeze(1)


class SourceAwareOneHotMLP(nn.Module):
    def __init__(self, input_dim: int, n_sources: int, hidden_dims: Tuple[int, ...], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim + n_sources
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        self.backbone = nn.Sequential(*layers)
        self.output = nn.Linear(prev_dim, 1)

    def forward(self, x_num: torch.Tensor, source_idx=None, source_onehot: torch.Tensor = None):
        x = torch.cat([x_num, source_onehot], dim=1)
        h = self.backbone(x)
        return self.output(h).squeeze(1)


# ============================================================
# metrics
# ============================================================

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)
    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan

    return {
        "auc": auc,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "logloss": log_loss(y_true, y_prob, labels=[0, 1]),
        "brier": brier_score_loss(y_true, y_prob),
    }


# ============================================================
# evaluation
# ============================================================

def evaluate_model(model: nn.Module, loader: DataLoader, mode: str):
    model.eval()
    all_probs = []
    all_targets = []
    total_loss = 0.0
    total_n = 0

    with torch.no_grad():
        for x_num, source_idx, source_onehot, y in loader:
            x_num = x_num.to(CFG.device)
            source_idx = source_idx.to(CFG.device)
            source_onehot = source_onehot.to(CFG.device)
            y = y.to(CFG.device)

            if mode == "agnostic":
                logits = model(x_num)
            elif mode == "embedding":
                logits = model(x_num, source_idx)
            elif mode == "onehot":
                logits = model(x_num, source_onehot=source_onehot)
            else:
                raise ValueError("Unknown mode.")

            probs = torch.sigmoid(logits)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, y)

            total_loss += loss.item() * len(y)
            total_n += len(y)

            all_probs.append(probs.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    avg_loss = total_loss / total_n
    return avg_loss, y_true, y_prob


# ============================================================
# training
# ============================================================

def train_one_model(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    mode: str,
    pos_weight: torch.Tensor,
):
    model = model.to(CFG.device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(CFG.device))
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.learning_rate,
        weight_decay=CFG.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=CFG.min_lr,
    )

    active_unit = _CKPT.current_unit
    if active_unit is None:
        raise RuntimeError('Checkpoint unit was not set before training.')
    start_epoch, best_model_state, best_valid_loss, epochs_without_improvement, history = \
        _CKPT.restore_training(active_unit, model, optimizer, scheduler)

    if epochs_without_improvement >= CFG.patience:
        print(f'[CHECKPOINT] Early stopping reached before interruption: {active_unit}.')
    for epoch in range(start_epoch if epochs_without_improvement < CFG.patience else CFG.max_epochs,
                       CFG.max_epochs):
        model.train()
        total_train_loss = 0.0
        total_train_n = 0

        for x_num, source_idx, source_onehot, y in train_loader:
            x_num = x_num.to(CFG.device)
            source_idx = source_idx.to(CFG.device)
            source_onehot = source_onehot.to(CFG.device)
            y = y.to(CFG.device)

            optimizer.zero_grad()

            if mode == "agnostic":
                logits = model(x_num)
            elif mode == "embedding":
                logits = model(x_num, source_idx)
            elif mode == "onehot":
                logits = model(x_num, source_onehot=source_onehot)
            else:
                raise ValueError("Unknown mode.")

            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_train_loss += loss.item() * len(y)
            total_train_n += len(y)

        train_loss = total_train_loss / total_train_n
        valid_loss, _, _ = evaluate_model(model, valid_loader, mode=mode)
        scheduler.step(valid_loss)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "lr": optimizer.param_groups[0]["lr"],
        })

        if valid_loss < best_valid_loss - 1e-5:
            best_valid_loss = valid_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        _CKPT.save_epoch_if_due(active_unit, epoch, model, optimizer, scheduler,
                                best_model_state, best_valid_loss, epochs_without_improvement, history)
        if epochs_without_improvement >= CFG.patience:
            break

    model.load_state_dict(best_model_state)
    return model, pd.DataFrame(history)


# ============================================================
# model factory
# ============================================================

def instantiate_model(
    model_name: str,
    input_dim: int,
    n_sources: int,
    embedding_dim: Optional[int] = None,
):
    if model_name == "source_agnostic_mlp":
        return SourceAgnosticMLP(
            input_dim=input_dim,
            hidden_dims=CFG.hidden_dims,
            dropout=CFG.dropout,
        ), "agnostic"

    if model_name == "source_aware_embedding_mlp":
        if embedding_dim is None:
            raise ValueError("embedding_dim must be provided for source_aware_embedding_mlp.")
        return SourceAwareEmbeddingMLP(
            input_dim=input_dim,
            n_sources=n_sources,
            source_embedding_dim=embedding_dim,
            hidden_dims=CFG.hidden_dims,
            dropout=CFG.dropout,
        ), "embedding"

    if model_name == "source_aware_onehot_mlp":
        return SourceAwareOneHotMLP(
            input_dim=input_dim,
            n_sources=n_sources,
            hidden_dims=CFG.hidden_dims,
            dropout=CFG.dropout,
        ), "onehot"

    raise ValueError("Unknown model name.")


def full_model_name(base_model_name: str, embedding_dim: Optional[int]) -> str:
    if base_model_name == "source_aware_embedding_mlp":
        return f"{base_model_name}_m{embedding_dim}"
    return base_model_name


# ============================================================
# source-wise metrics
# ============================================================

def sourcewise_metrics(
    y_true: np.ndarray,
    y_prob: np.ndarray,
    source_names: np.ndarray,
    model_name: str,
    split_id: str,
):
    rows = []
    for src in np.unique(source_names):
        idx = (source_names == src)
        yt = y_true[idx]
        yp = y_prob[idx]

        if len(yt) < 5:
            continue

        met = compute_metrics(yt, yp)
        met["source"] = src
        met["model"] = model_name
        met["split_id"] = split_id
        met["n"] = len(yt)
        rows.append(met)
    return rows


# ============================================================
# CV pipeline
# ============================================================

def run_repeated_cv(
    X: np.ndarray,
    y: np.ndarray,
    source_idx: np.ndarray,
    source_onehot: np.ndarray,
    source_names: np.ndarray,
    base_model_name: str,
    embedding_dim: Optional[int] = None,
):
    rskf = RepeatedStratifiedKFold(
        n_splits=CFG.n_splits,
        n_repeats=CFG.n_repeats,
        random_state=CFG.random_seed,
    )

    model_name = full_model_name(base_model_name, embedding_dim)

    fold_results = []
    oof_rows = []
    source_rows = []
    histories = []

    n_sources = source_onehot.shape[1]
    split_counter = 0

    for train_val_idx, test_idx in rskf.split(X, y):
        split_counter += 1
        unit_key = f"CV|{model_name}|split_{split_counter}"
        cached = _CKPT.get_completed(unit_key)
        if cached is not None:
            print(f"[RESUME] Skipping finished {unit_key}")
            fold_results.extend(cached['metrics'])
            oof_rows.extend(cached['predictions'])
            source_rows.extend(cached['source_metrics'])
            histories.append(pd.DataFrame(cached['history']))
            continue
        _CKPT.current_unit = unit_key
        print(f"\n[{model_name}] split {split_counter}/{CFG.n_splits * CFG.n_repeats}")

        X_train_val = X[train_val_idx]
        y_train_val = y[train_val_idx]
        s_train_val = source_idx[train_val_idx]
        s1_train_val = source_onehot[train_val_idx]

        X_test = X[test_idx]
        y_test = y[test_idx]
        s_test = source_idx[test_idx]
        s1_test = source_onehot[test_idx]
        source_names_test = source_names[test_idx]

        tr_idx, va_idx = train_test_split(
            np.arange(len(y_train_val)),
            test_size=CFG.valid_size,
            stratify=y_train_val,
            random_state=CFG.random_seed + split_counter,
        )

        X_train = X_train_val[tr_idx]
        y_train = y_train_val[tr_idx]
        s_train = s_train_val[tr_idx]
        s1_train = s1_train_val[tr_idx]

        X_valid = X_train_val[va_idx]
        y_valid = y_train_val[va_idx]
        s_valid = s_train_val[va_idx]
        s1_valid = s1_train_val[va_idx]

        X_train, X_valid, X_test = foldwise_median_impute_and_scale(
            X_train, X_valid, X_test
        )

        train_loader, valid_loader = make_loaders(
            X_train, s_train, s1_train, y_train,
            X_valid, s_valid, s1_valid, y_valid,
        )

        n_pos = y_train.sum()
        n_neg = len(y_train) - n_pos
        pos_weight = torch.tensor(n_neg / max(n_pos, 1.0), dtype=torch.float32)

        model, mode = instantiate_model(
            model_name=base_model_name,
            input_dim=X.shape[1],
            n_sources=n_sources,
            embedding_dim=embedding_dim,
        )

        model, history_df = train_one_model(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            mode=mode,
            pos_weight=pos_weight,
        )

        history_df["model"] = model_name
        history_df["split_id"] = split_counter
        histories.append(history_df)
        oof_start = len(oof_rows)
        source_start = len(source_rows)

        test_ds = TabularDataset(X_test, s_test, s1_test, y_test)
        test_loader = DataLoader(
            test_ds,
            batch_size=CFG.batch_size,
            shuffle=False,
            num_workers=CFG.num_workers,
        )

        _, y_true, y_prob = evaluate_model(model, test_loader, mode=mode)
        metrics = compute_metrics(y_true, y_prob)
        metrics["model"] = model_name
        metrics["base_model"] = base_model_name
        metrics["embedding_dim"] = embedding_dim if embedding_dim is not None else np.nan
        metrics["split_id"] = split_counter
        fold_results.append(metrics)

        for i, idx in enumerate(test_idx):
            oof_rows.append({
                "row_id": int(idx),
                "split_id": split_counter,
                "model": model_name,
                "base_model": base_model_name,
                "embedding_dim": embedding_dim if embedding_dim is not None else np.nan,
                "y_true": int(y_true[i]),
                "y_prob": float(y_prob[i]),
                "source": source_names_test[i],
            })

        src_rows = sourcewise_metrics(
            y_true=y_true,
            y_prob=y_prob,
            source_names=source_names_test,
            model_name=model_name,
            split_id=split_counter,
        )
        for row in src_rows:
            row["base_model"] = base_model_name
            row["embedding_dim"] = embedding_dim if embedding_dim is not None else np.nan
        source_rows.extend(src_rows)
        _CKPT.complete(unit_key, metrics=[dict(metrics)], predictions=oof_rows[oof_start:],
                       source_metrics=source_rows[source_start:],
                       history=history_df.to_dict('records'))
        _CKPT.current_unit = None

        print(
            f"AUC={metrics['auc']:.4f}, "
            f"ACC={metrics['accuracy']:.4f}, "
            f"BACC={metrics['balanced_accuracy']:.4f}, "
            f"F1={metrics['f1']:.4f}, "
            f"LogLoss={metrics['logloss']:.4f}"
        )

    return (
        pd.DataFrame(fold_results),
        pd.DataFrame(oof_rows),
        pd.DataFrame(source_rows),
        pd.concat(histories, axis=0, ignore_index=True),
    )


# ============================================================
# summaries
# ============================================================

def summarize_results(df_results: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["auc", "accuracy", "balanced_accuracy", "f1", "logloss", "brier"]
    rows = []

    for model_name, sub in df_results.groupby("model"):
        row = {
            "model": model_name,
            "base_model": sub["base_model"].iloc[0],
            "embedding_dim": sub["embedding_dim"].iloc[0],
        }
        for col in metric_cols:
            row[f"{col}_mean"] = sub[col].mean()
            row[f"{col}_sd"] = sub[col].std(ddof=1)
        rows.append(row)

    return pd.DataFrame(rows)


def summarize_source_results(df_source: pd.DataFrame) -> pd.DataFrame:
    metric_cols = ["auc", "accuracy", "balanced_accuracy", "f1", "logloss", "brier"]
    rows = []

    for (model_name, src), sub in df_source.groupby(["model", "source"]):
        row = {
            "model": model_name,
            "base_model": sub["base_model"].iloc[0],
            "embedding_dim": sub["embedding_dim"].iloc[0],
            "source": src,
        }
        for col in metric_cols:
            row[f"{col}_mean"] = sub[col].mean()
            row[f"{col}_sd"] = sub[col].std(ddof=1)
        row["n_mean"] = sub["n"].mean()
        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# plotting
# ============================================================

def prettify_model_name(name: str) -> str:
    if name == "source_agnostic_mlp":
        return "Source-agnostic"
    if name == "source_aware_onehot_mlp":
        return "Source-aware one-hot"
    if name.startswith("source_aware_embedding_mlp_m"):
        m = name.split("_m")[-1]
        return f"Embedding m={m}"
    return name


def save_barplot_with_errorbars(
    df_fold: pd.DataFrame,
    models_to_plot: List[str],
    metric: str,
    ylabel: str,
    filename: str,
    higher_is_better: bool = True,
):
    sub = df_fold[df_fold["model"].isin(models_to_plot)].copy()
    summary = (
        sub.groupby("model")[metric]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("mean", ascending=not higher_is_better)
    )

    x = np.arange(len(summary))

    plt.figure(figsize=(9, 5))
    plt.bar(x, summary["mean"].values)
    plt.errorbar(
        x,
        summary["mean"].values,
        yerr=summary["std"].values,
        fmt="none",
        capsize=5,
    )
    plt.xticks(x, [prettify_model_name(m) for m in summary["model"]], rotation=20)
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} across cross-validation splits")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_roc_plot(df_oof: pd.DataFrame, models_to_plot: List[str], filename: str):
    plt.figure(figsize=(7, 5))

    sub_all = df_oof[df_oof["model"].isin(models_to_plot)].copy()
    for model_name, sub in sub_all.groupby("model"):
        fpr, tpr, _ = roc_curve(sub["y_true"].values, sub["y_prob"].values)
        auc = roc_auc_score(sub["y_true"].values, sub["y_prob"].values)
        plt.plot(fpr, tpr, label=f"{prettify_model_name(model_name)} (AUC={auc:.3f})")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("Out-of-fold ROC curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_calibration_plot(df_oof: pd.DataFrame, models_to_plot: List[str], filename: str, n_bins: int = 10):
    plt.figure(figsize=(7, 5))

    sub_all = df_oof[df_oof["model"].isin(models_to_plot)].copy()
    for model_name, sub in sub_all.groupby("model"):
        frac_pos, mean_pred = calibration_curve(
            sub["y_true"].values,
            sub["y_prob"].values,
            n_bins=n_bins,
            strategy="quantile",
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=prettify_model_name(model_name))

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration curves")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_embedding_dimension_curve(
    df_summary: pd.DataFrame,
    metric: str,
    ylabel: str,
    filename: str,
    higher_is_better: bool = True,
):
    sub = df_summary[df_summary["base_model"] == "source_aware_embedding_mlp"].copy()
    sub = sub.sort_values("embedding_dim")

    plt.figure(figsize=(7, 5))
    plt.errorbar(
        sub["embedding_dim"].values,
        sub[f"{metric}_mean"].values,
        yerr=sub[f"{metric}_sd"].values,
        marker="o",
        linewidth=2,
        capsize=4,
    )
    plt.xscale("log", base=2)
    plt.xlabel("Embedding dimension")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} as a function of embedding dimension")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_sourcewise_gain_plot(
    df_source_summary: pd.DataFrame,
    metric: str,
    baseline_model: str,
    comparator_model: str,
    filename: str,
):
    base = (
        df_source_summary[df_source_summary["model"] == baseline_model][["source", f"{metric}_mean"]]
        .rename(columns={f"{metric}_mean": "baseline"})
    )
    comp = (
        df_source_summary[df_source_summary["model"] == comparator_model][["source", f"{metric}_mean"]]
        .rename(columns={f"{metric}_mean": "comparator"})
    )

    merged = base.merge(comp, on="source", how="inner")
    merged["gain"] = merged["comparator"] - merged["baseline"]
    merged = merged.sort_values("gain")

    x = np.arange(len(merged))

    plt.figure(figsize=(8, 5))
    plt.bar(x, merged["gain"].values)
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.xticks(x, merged["source"].values, rotation=25)
    plt.ylabel(f"{metric} gain")
    plt.title(
        f"{prettify_model_name(comparator_model)} minus "
        f"{prettify_model_name(baseline_model)} by source"
    )
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_training_curve(df_history: pd.DataFrame, model_name: str, filename: str):
    sub = df_history[df_history["model"] == model_name].copy()
    if sub.empty:
        return

    avg = sub.groupby("epoch")[["train_loss", "valid_loss"]].mean().reset_index()

    plt.figure(figsize=(7, 5))
    plt.plot(avg["epoch"], avg["train_loss"], label="Train loss")
    plt.plot(avg["epoch"], avg["valid_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training curves: {prettify_model_name(model_name)}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


# ============================================================
# helper to find best embedding dimension
# ============================================================

def get_best_embedding_model_name(df_summary: pd.DataFrame, metric: str = "auc_mean", maximize: bool = True) -> str:
    sub = df_summary[df_summary["base_model"] == "source_aware_embedding_mlp"].copy()
    sub = sub.sort_values(metric, ascending=not maximize)
    return sub.iloc[0]["model"]


# ============================================================
# main
# ============================================================

def main():
    df, X, y, source_idx, source_onehot, feature_cols, source_encoder = load_and_prepare_data(CFG.data_path)
    source_names = source_encoder.inverse_transform(source_idx)
    _CKPT.register('CV', CFG, df)

    print("Dataset shape:", df.shape)
    print("Feature columns:", feature_cols)
    print("Source levels:", list(source_encoder.classes_))
    print("Embedding dimensions explored:", CFG.embedding_dims)
    print("Device:", CFG.device)

    run_specs = [("source_agnostic_mlp", None)]
    run_specs += [("source_aware_embedding_mlp", m) for m in CFG.embedding_dims]
    run_specs += [("source_aware_onehot_mlp", None)]

    all_fold_results = []
    all_oof = []
    all_source = []
    all_histories = []

    for base_model_name, embedding_dim in run_specs:
        fold_df, oof_df, source_df, hist_df = run_repeated_cv(
            X=X,
            y=y,
            source_idx=source_idx,
            source_onehot=source_onehot,
            source_names=source_names,
            base_model_name=base_model_name,
            embedding_dim=embedding_dim,
        )
        all_fold_results.append(fold_df)
        all_oof.append(oof_df)
        all_source.append(source_df)
        all_histories.append(hist_df)

    df_results = pd.concat(all_fold_results, axis=0, ignore_index=True)
    df_oof = pd.concat(all_oof, axis=0, ignore_index=True)
    df_source = pd.concat(all_source, axis=0, ignore_index=True)
    df_history = pd.concat(all_histories, axis=0, ignore_index=True)

    summary = summarize_results(df_results)
    source_summary = summarize_source_results(df_source)

    # save tables
    df_results.to_csv(os.path.join(TABLE_DIR, "cv_fold_results.csv"), index=False)
    df_oof.to_csv(os.path.join(TABLE_DIR, "oof_predictions.csv"), index=False)
    df_source.to_csv(os.path.join(TABLE_DIR, "sourcewise_fold_results.csv"), index=False)
    df_history.to_csv(os.path.join(TABLE_DIR, "training_histories.csv"), index=False)
    summary.to_csv(os.path.join(TABLE_DIR, "cv_summary.csv"), index=False)
    source_summary.to_csv(os.path.join(TABLE_DIR, "sourcewise_summary.csv"), index=False)

    # choose best embedding model by AUC
    best_embedding_model = get_best_embedding_model_name(summary, metric="auc_mean", maximize=True)
    print(f"\nBest embedding model by mean AUC: {best_embedding_model}")

    # show only source-agnostic and best embedding model in graphs
    core_models = ["source_agnostic_mlp", best_embedding_model]

    # core comparison figures
    save_barplot_with_errorbars(df_results, core_models, "auc", "AUC", "figure_auc_cv.png", higher_is_better=True)
    save_barplot_with_errorbars(
        df_results, core_models, "balanced_accuracy", "Balanced accuracy",
        "figure_balanced_accuracy_cv.png", higher_is_better=True
    )
    save_barplot_with_errorbars(df_results, core_models, "f1", "F1 score", "figure_f1_cv.png", higher_is_better=True)
    save_barplot_with_errorbars(df_results, core_models, "logloss", "Log-loss", "figure_logloss_cv.png", higher_is_better=False)

    save_roc_plot(df_oof, core_models, "figure_roc_oof.png")
    save_calibration_plot(df_oof, core_models, "figure_calibration_oof.png", n_bins=10)

    # embedding sweep figures
    save_embedding_dimension_curve(summary, "auc", "AUC", "figure_embedding_dimension_auc.png", higher_is_better=True)
    save_embedding_dimension_curve(
        summary, "balanced_accuracy", "Balanced accuracy",
        "figure_embedding_dimension_bacc.png", higher_is_better=True
    )
    save_embedding_dimension_curve(
        summary, "logloss", "Log-loss",
        "figure_embedding_dimension_logloss.png", higher_is_better=False
    )

    # source-wise gains for best embedding model
    save_sourcewise_gain_plot(
        df_source_summary=source_summary,
        metric="auc",
        baseline_model="source_agnostic_mlp",
        comparator_model=best_embedding_model,
        filename="figure_sourcewise_auc_best_embedding_vs_agnostic.png",
    )
    save_sourcewise_gain_plot(
        df_source_summary=source_summary,
        metric="balanced_accuracy",
        baseline_model="source_agnostic_mlp",
        comparator_model=best_embedding_model,
        filename="figure_sourcewise_bacc_best_embedding_vs_agnostic.png",
    )

    # training curve for best embedding model
    save_training_curve(df_history, best_embedding_model, "figure_training_curve_best_embedding.png")

    print("\nSummary:")
    print(summary.sort_values("auc_mean", ascending=False))

    print("\nSaved tables in:")
    print(TABLE_DIR)

    print("\nSaved figures in:")
    print(FIG_DIR)


main()

# ============= EXTRA CALIBRATION FIGURES =============
import os
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.calibration import calibration_curve

OUTPUT_DIR = os.path.join(RESULTS_PARENT, "heart_disease_application_embedding_sweep")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def prettify_model_name(name: str) -> str:
    if name == "source_agnostic_mlp":
        return "Source-agnostic"
    if name.startswith("source_aware_embedding_mlp_m"):
        m = name.split("_m")[-1]
        return f"Embedding m={m}"
    return name

def prettify_source_name(src: str) -> str:
    src = str(src).replace("_", " ")
    return src.title()

def safe_filename(text: str) -> str:
    return re.sub(r"[^A-Za-z0-9_-]+", "_", str(text)).strip("_")

def save_improved_calibration_plot_by_source(
    df_oof: pd.DataFrame,
    models_to_plot,
    filename_prefix: str = "figure_calibration_by_source_improved",
    n_bins: int = 5,
):
    sub_all = df_oof[df_oof["model"].isin(models_to_plot)].copy()

    if sub_all.empty:
        print("No matching models found in df_oof.")
        return

    for src, sub_src in sub_all.groupby("source"):
        if len(sub_src) == 0:
            continue

        fig = plt.figure(figsize=(7, 7))
        gs = fig.add_gridspec(2, 1, height_ratios=[4, 1], hspace=0.08)

        ax1 = fig.add_subplot(gs[0])
        ax2 = fig.add_subplot(gs[1], sharex=ax1)

        plotted_any = False

        for model_name, sub in sub_src.groupby("model"):
            y_true = sub["y_true"].values
            y_prob = sub["y_prob"].values

            if len(np.unique(y_true)) < 2:
                continue

            frac_pos, mean_pred = calibration_curve(
                y_true,
                y_prob,
                n_bins=n_bins,
                strategy="quantile",
            )

            ax1.plot(
                mean_pred,
                frac_pos,
                marker="o",
                linewidth=2,
                markersize=6,
                label=prettify_model_name(model_name),
            )

            ax2.hist(
                y_prob,
                bins=np.linspace(0, 1, n_bins + 1),
                alpha=0.4,
                label=prettify_model_name(model_name),
            )

            plotted_any = True

        if not plotted_any:
            plt.close(fig)
            continue

        ax1.plot([0, 1], [0, 1], linestyle="--", linewidth=1.5)
        ax1.set_ylabel("Observed frequency")
        ax1.set_xlim(0, 1)
        ax1.set_ylim(0, 1.02)
        ax1.legend(frameon=True)

        ax1.set_title(
            f"Calibration by source: {prettify_source_name(src)} (n={sub_src.row_id.nunique()})"
        )

        ax2.set_xlabel("Predicted probability")
        ax2.set_ylabel("Count")
        ax2.set_xlim(0, 1)

        safe_src = safe_filename(src)
        outpath = os.path.join(FIG_DIR, f"{filename_prefix}_{safe_src}.png")
        plt.tight_layout()
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"Saved: {outpath}")

# load saved OOF predictions
df_oof = pd.read_csv(os.path.join(TABLE_DIR, "oof_predictions.csv"))

# choose best embedding automatically from cv_summary.csv
df_summary = pd.read_csv(os.path.join(TABLE_DIR, "cv_summary.csv"))
best_embedding_model = (
    df_summary[df_summary["base_model"] == "source_aware_embedding_mlp"]
    .sort_values("auc_mean", ascending=False)
    .iloc[0]["model"]
)

models_to_plot = [
    "source_agnostic_mlp",
    best_embedding_model,
]

save_improved_calibration_plot_by_source(
    df_oof=df_oof,
    models_to_plot=models_to_plot,
    filename_prefix="figure_calibration_by_source_improved",
    n_bins=4,
)

# ============= SOURCE HETEROGENEITY TABLES =============
import os
import numpy as np
import pandas as pd

DATA_PATH = "heart_disease.csv"
OUTPUT_DIR = os.path.join(RESULTS_PARENT, "heart_disease_application_embedding_sweep")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
os.makedirs(TABLE_DIR, exist_ok=True)

def summarize_sources_descriptively(df: pd.DataFrame, feature_cols):
    rows = []
    overall_prev = df["target"].mean()

    for src, sub in df.groupby("source"):
        row = {
            "source": src,
            "n": len(sub),
            "target_prevalence": sub["target"].mean(),
            "prevalence_diff_from_overall": sub["target"].mean() - overall_prev,
        }
        for col in feature_cols:
            row[f"{col}_mean"] = sub[col].mean()
            row[f"{col}_sd"] = sub[col].std(ddof=1)
        rows.append(row)

    return pd.DataFrame(rows)

def standardized_mean_differences_source_vs_rest(df: pd.DataFrame, feature_cols):
    rows = []

    for src in sorted(df["source"].astype(str).unique()):
        src_df = df[df["source"].astype(str) == src]
        rest_df = df[df["source"].astype(str) != src]

        for col in feature_cols:
            m1 = src_df[col].mean()
            m0 = rest_df[col].mean()
            s1 = src_df[col].std(ddof=1)
            s0 = rest_df[col].std(ddof=1)

            pooled_sd = np.sqrt((s1**2 + s0**2) / 2.0)
            smd = (m1 - m0) / pooled_sd if pooled_sd > 0 else np.nan

            rows.append({
                "source": src,
                "feature": col,
                "mean_source": m1,
                "mean_rest": m0,
                "sd_source": s1,
                "sd_rest": s0,
                "smd": smd,
                "abs_smd": abs(smd) if pd.notna(smd) else np.nan,
            })

    return pd.DataFrame(rows)

def summarize_top_heterogeneity(df_smd: pd.DataFrame, top_k: int = 5):
    rows = []
    for src, sub in df_smd.groupby("source"):
        top = sub.sort_values("abs_smd", ascending=False).head(top_k).copy()
        top["rank_within_source"] = np.arange(1, len(top) + 1)
        rows.append(top)
    return pd.concat(rows, axis=0, ignore_index=True)

# load data
df = load_clean_heart_dataframe(DATA_PATH)

if "target" not in df.columns or "source" not in df.columns:
    raise ValueError("The dataset must contain 'target' and 'source' columns.")

feature_cols = [c for c in df.columns if c not in ["target", "source"]]

# compute tables
df_source_desc = summarize_sources_descriptively(df, feature_cols)
df_smd = standardized_mean_differences_source_vs_rest(df, feature_cols)
df_top_smd = summarize_top_heterogeneity(df_smd, top_k=5)

# save outputs
df_source_desc.to_csv(os.path.join(TABLE_DIR, "source_descriptives.csv"), index=False)
df_smd.to_csv(os.path.join(TABLE_DIR, "source_standardized_mean_differences.csv"), index=False)
df_top_smd.to_csv(os.path.join(TABLE_DIR, "source_top_heterogeneity_features.csv"), index=False)

print("Saved:")
print(os.path.join(TABLE_DIR, "source_descriptives.csv"))
print(os.path.join(TABLE_DIR, "source_standardized_mean_differences.csv"))
print(os.path.join(TABLE_DIR, "source_top_heterogeneity_features.csv"))

print("\nSource descriptives:")
print(df_source_desc)

print("\nTop heterogeneous features by source:")
print(df_top_smd[["source", "rank_within_source", "feature", "smd", "abs_smd"]])

# ============= SOURCEWISE BRIER GAIN FIGURE =============
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

OUTPUT_DIR = os.path.join(RESULTS_PARENT, "heart_disease_application_embedding_sweep")
TABLE_DIR = os.path.join(OUTPUT_DIR, "tables")
FIG_DIR = os.path.join(OUTPUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

def prettify_model_name(name: str) -> str:
    if name == "source_agnostic_mlp":
        return "Source-agnostic"
    if name.startswith("source_aware_embedding_mlp_m"):
        m = name.split("_m")[-1]
        return f"Embedding m={m}"
    return name

def save_sourcewise_brier_gain_plot(
    df_source_summary: pd.DataFrame,
    baseline_model: str,
    comparator_model: str,
    filename: str = "figure_sourcewise_brier_gain.png",
):
    base = (
        df_source_summary[df_source_summary["model"] == baseline_model][["source", "brier_mean"]]
        .rename(columns={"brier_mean": "baseline_brier"})
    )
    comp = (
        df_source_summary[df_source_summary["model"] == comparator_model][["source", "brier_mean"]]
        .rename(columns={"brier_mean": "comparator_brier"})
    )

    merged = base.merge(comp, on="source", how="inner")
    merged["brier_gain"] = merged["baseline_brier"] - merged["comparator_brier"]
    merged = merged.sort_values("brier_gain", ascending=True)

    x = np.arange(len(merged))

    plt.figure(figsize=(8, 5))
    plt.bar(x, merged["brier_gain"].values)
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.xticks(x, merged["source"].values, rotation=25)
    plt.ylabel("Brier score reduction")
    plt.title(
        f"Brier gain by source\n"
        f"{prettify_model_name(comparator_model)} vs {prettify_model_name(baseline_model)}"
    )
    plt.tight_layout()
    outpath = os.path.join(FIG_DIR, filename)
    plt.savefig(outpath, dpi=300, bbox_inches="tight")
    plt.close()

    print(f"Saved: {outpath}")
    print("\nSourcewise Brier gains:")
    print(merged[["source", "baseline_brier", "comparator_brier", "brier_gain"]])

# load sourcewise summary
df_source_summary = pd.read_csv(os.path.join(TABLE_DIR, "sourcewise_summary.csv"))

# automatically choose best embedding model from cv_summary.csv
df_summary = pd.read_csv(os.path.join(TABLE_DIR, "cv_summary.csv"))
best_embedding_model = (
    df_summary[df_summary["base_model"] == "source_aware_embedding_mlp"]
    .sort_values("auc_mean", ascending=False)
    .iloc[0]["model"]
)

save_sourcewise_brier_gain_plot(
    df_source_summary=df_source_summary,
    baseline_model="source_agnostic_mlp",
    comparator_model=best_embedding_model,
    filename="figure_sourcewise_brier_gain_best_embedding_vs_agnostic.png",
)

# ============= STUDY 2: LOSO + ALL EXISTING FIGURES =============
"""
Complete application pipeline for the uploaded heart_disease.csv dataset.

This version keeps only leave-one-source-out evaluation and compares:
1. source_agnostic_mlp
2. source_aware_embedding_mlp_m{m} for m in {4,8,16,32,64,128}
3. source_aware_onehot_mlp

Evaluation design
-----------------
For each source:
- hold that source out entirely as test data
- train on the remaining sources
- create an internal stratified validation split from the training sources
- use early stopping on validation loss

Unseen-source handling
----------------------
- source_agnostic_mlp: no issue
- source_aware_onehot_mlp: test source represented by all-zero source indicators
- source_aware_embedding_mlp: test source represented by the mean training-source embedding

Training
--------
- AdamW
- early stopping
- ReduceLROnPlateau
- gradient clipping
- batch normalization
- dropout
- class-weighted loss

Extra performance measures
--------------------------
- AUC
- accuracy
- balanced accuracy
- F1
- precision
- recall
- specificity
- sensitivity
- negative predictive value
- MCC
- log-loss
- Brier score
- PR-AUC

Expected dataset columns
------------------------
age, sex, cp, trestbps, fbs, restecg, thalach,
exang, oldpeak, slope, ca, thal, target, source
(cholesterol is intentionally excluded)
"""

import os
import copy
import random
from dataclasses import dataclass
from typing import Dict, Tuple, Optional, List

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    log_loss,
    brier_score_loss,
    precision_score,
    recall_score,
    matthews_corrcoef,
    average_precision_score,
    confusion_matrix,
    roc_curve,
    precision_recall_curve,
)
from sklearn.calibration import calibration_curve

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader


# ============================================================
# configuration
# ============================================================

@dataclass
class Config:
    data_path: str = DATA_PATH_OVERRIDE
    output_dir: str = os.path.join(RESULTS_PARENT, "heart_disease_application_loso_only")

    random_seed: int = 6
    valid_size: float = 0.4

    batch_size: int = 8
    max_epochs: int = 100
    patience: int = 10

    learning_rate: float = 1e-2
    weight_decay: float = 1e-4
    min_lr: float = 1e-6
    dropout: float = 0.25
    hidden_dims: Tuple[int, ...] = (64, 32)

    embedding_dims: Tuple[int, ...] = (2, 4, 8, 16, 32, 64, 128, 256)

    num_workers: int = 0
    device: str = "cuda" if torch.cuda.is_available() else "cpu"


CFG = Config()
os.makedirs(CFG.output_dir, exist_ok=True)

TABLE_DIR = os.path.join(CFG.output_dir, "tables")
FIG_DIR = os.path.join(CFG.output_dir, "figures")
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)


# ============================================================
# reproducibility
# ============================================================

def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(CFG.random_seed)


# ============================================================
# dataset utilities
# ============================================================

class TabularDataset(Dataset):
    def __init__(
        self,
        x_num: np.ndarray,
        source_idx: np.ndarray,
        source_onehot: np.ndarray,
        y: np.ndarray,
    ):
        self.x_num = torch.tensor(x_num, dtype=torch.float32)
        self.source_idx = torch.tensor(source_idx, dtype=torch.long)
        self.source_onehot = torch.tensor(source_onehot, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32)

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return (
            self.x_num[idx],
            self.source_idx[idx],
            self.source_onehot[idx],
            self.y[idx],
        )


def load_and_prepare_data(path: str):
    df = load_clean_heart_dataframe(path)

    if "target" not in df.columns:
        raise ValueError("The dataset must contain a 'target' column.")
    if "source" not in df.columns:
        raise ValueError("The dataset must contain a 'source' column.")

    feature_cols = [c for c in df.columns if c not in ["target", "source"]]
    X = df[feature_cols].astype(float).values
    y = df["target"].astype(int).values
    source_raw = df["source"].astype(str).values

    source_encoder = LabelEncoder()
    source_idx = source_encoder.fit_transform(source_raw)
    n_sources = len(source_encoder.classes_)

    source_onehot = np.zeros((len(source_idx), n_sources), dtype=float)
    source_onehot[np.arange(len(source_idx)), source_idx] = 1.0

    return df, X, y, source_idx, source_onehot, feature_cols, source_encoder


def make_loaders(
    x_train: np.ndarray,
    s_train: np.ndarray,
    s1_train: np.ndarray,
    y_train: np.ndarray,
    x_valid: np.ndarray,
    s_valid: np.ndarray,
    s1_valid: np.ndarray,
    y_valid: np.ndarray,
):
    train_ds = TabularDataset(x_train, s_train, s1_train, y_train)
    valid_ds = TabularDataset(x_valid, s_valid, s1_valid, y_valid)

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG.batch_size,
        shuffle=True,
        num_workers=CFG.num_workers,
        drop_last=(len(train_ds) % CFG.batch_size == 1),
        pin_memory=torch.cuda.is_available(),
    )
    valid_loader = DataLoader(
        valid_ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )
    return train_loader, valid_loader


def make_single_loader(
    x: np.ndarray,
    s: np.ndarray,
    s1: np.ndarray,
    y: np.ndarray,
):
    ds = TabularDataset(x, s, s1, y)
    return DataLoader(
        ds,
        batch_size=CFG.batch_size,
        shuffle=False,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
    )


# ============================================================
# models
# ============================================================

class SourceAgnosticMLP(nn.Module):
    def __init__(self, input_dim: int, hidden_dims: Tuple[int, ...], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        self.backbone = nn.Sequential(*layers)
        self.output = nn.Linear(prev_dim, 1)

    def forward(self, x_num: torch.Tensor, source_idx=None, source_onehot=None):
        h = self.backbone(x_num)
        return self.output(h).squeeze(1)


class SourceAwareEmbeddingMLP(nn.Module):
    def __init__(
        self,
        input_dim: int,
        n_sources: int,
        source_embedding_dim: int,
        hidden_dims: Tuple[int, ...],
        dropout: float,
    ):
        super().__init__()
        self.source_embedding = nn.Embedding(n_sources, source_embedding_dim)

        layers = []
        prev_dim = input_dim + source_embedding_dim
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        self.backbone = nn.Sequential(*layers)
        self.output = nn.Linear(prev_dim, 1)

    def forward(
        self,
        x_num: torch.Tensor,
        source_idx: Optional[torch.Tensor],
        source_onehot=None,
        override_embedding: Optional[torch.Tensor] = None,
    ):
        if override_embedding is None:
            e = self.source_embedding(source_idx)
        else:
            e = override_embedding
        x = torch.cat([x_num, e], dim=1)
        h = self.backbone(x)
        return self.output(h).squeeze(1)

    def mean_training_embedding(self, train_source_ids: np.ndarray) -> torch.Tensor:
        unique_ids = np.unique(train_source_ids)
        weight = self.source_embedding.weight.detach()
        return weight[unique_ids].mean(dim=0)


class SourceAwareOneHotMLP(nn.Module):
    def __init__(self, input_dim: int, n_sources: int, hidden_dims: Tuple[int, ...], dropout: float):
        super().__init__()
        layers = []
        prev_dim = input_dim + n_sources
        for h in hidden_dims:
            layers.extend([
                nn.Linear(prev_dim, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ])
            prev_dim = h
        self.backbone = nn.Sequential(*layers)
        self.output = nn.Linear(prev_dim, 1)
        self.n_sources = n_sources

    def forward(self, x_num: torch.Tensor, source_idx=None, source_onehot: Optional[torch.Tensor] = None):
        x = torch.cat([x_num, source_onehot], dim=1)
        h = self.backbone(x)
        return self.output(h).squeeze(1)


# ============================================================
# metrics
# ============================================================

def compute_metrics(y_true: np.ndarray, y_prob: np.ndarray, threshold: float = 0.5) -> Dict[str, float]:
    y_pred = (y_prob >= threshold).astype(int)

    auc = roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan
    pr_auc = average_precision_score(y_true, y_prob) if len(np.unique(y_true)) > 1 else np.nan

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else np.nan
    specificity = tn / (tn + fp) if (tn + fp) > 0 else np.nan
    npv = tn / (tn + fn) if (tn + fn) > 0 else np.nan
    recall = sensitivity

    return {
        "auc": auc,
        "pr_auc": pr_auc,
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall,
        "sensitivity": sensitivity,
        "specificity": specificity,
        "npv": npv,
        "mcc": matthews_corrcoef(y_true, y_pred),
        "logloss": log_loss(y_true, y_prob, labels=[0, 1]),
        "brier": brier_score_loss(y_true, y_prob),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
    }


# ============================================================
# evaluation
# ============================================================

def evaluate_model(model: nn.Module, loader: DataLoader, mode: str):
    model.eval()
    all_probs = []
    all_targets = []
    total_loss = 0.0
    total_n = 0

    with torch.no_grad():
        for x_num, source_idx, source_onehot, y in loader:
            x_num = x_num.to(CFG.device)
            source_idx = source_idx.to(CFG.device)
            source_onehot = source_onehot.to(CFG.device)
            y = y.to(CFG.device)

            if mode == "agnostic":
                logits = model(x_num)
            elif mode == "embedding":
                logits = model(x_num, source_idx)
            elif mode == "onehot":
                logits = model(x_num, source_onehot=source_onehot)
            else:
                raise ValueError("Unknown mode.")

            probs = torch.sigmoid(logits)
            loss = nn.functional.binary_cross_entropy_with_logits(logits, y)

            total_loss += loss.item() * len(y)
            total_n += len(y)

            all_probs.append(probs.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    avg_loss = total_loss / total_n
    return avg_loss, y_true, y_prob


def evaluate_embedding_model_unseen_source(
    model: SourceAwareEmbeddingMLP,
    loader: DataLoader,
    override_embedding_vector: torch.Tensor,
):
    model.eval()
    all_probs = []
    all_targets = []

    with torch.no_grad():
        for x_num, _, _, y in loader:
            x_num = x_num.to(CFG.device)
            y = y.to(CFG.device)

            batch_override = override_embedding_vector.unsqueeze(0).repeat(x_num.shape[0], 1)
            logits = model(x_num, source_idx=None, override_embedding=batch_override)
            probs = torch.sigmoid(logits)

            all_probs.append(probs.cpu().numpy())
            all_targets.append(y.cpu().numpy())

    y_prob = np.concatenate(all_probs)
    y_true = np.concatenate(all_targets)
    return y_true, y_prob


def evaluate_onehot_model_unseen_source(
    model: SourceAwareOneHotMLP,
    x: np.ndarray,
    y: np.ndarray,
):
    model.eval()
    x_t = torch.tensor(x, dtype=torch.float32, device=CFG.device)
    zero_onehot = torch.zeros((len(x), model.n_sources), dtype=torch.float32, device=CFG.device)

    with torch.no_grad():
        logits = model(x_t, source_onehot=zero_onehot)
        probs = torch.sigmoid(logits).cpu().numpy()

    return y, probs


# ============================================================
# training
# ============================================================

def train_one_model(
    model: nn.Module,
    train_loader: DataLoader,
    valid_loader: DataLoader,
    mode: str,
    pos_weight: torch.Tensor,
):
    model = model.to(CFG.device)

    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight.to(CFG.device))
    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG.learning_rate,
        weight_decay=CFG.weight_decay,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer,
        mode="min",
        factor=0.5,
        patience=8,
        min_lr=CFG.min_lr,
    )

    active_unit = _CKPT.current_unit
    if active_unit is None:
        raise RuntimeError('Checkpoint unit was not set before training.')
    start_epoch, best_model_state, best_valid_loss, epochs_without_improvement, history = \
        _CKPT.restore_training(active_unit, model, optimizer, scheduler)

    if epochs_without_improvement >= CFG.patience:
        print(f'[CHECKPOINT] Early stopping reached before interruption: {active_unit}.')
    for epoch in range(start_epoch if epochs_without_improvement < CFG.patience else CFG.max_epochs,
                       CFG.max_epochs):
        model.train()
        total_train_loss = 0.0
        total_train_n = 0

        for x_num, source_idx, source_onehot, y in train_loader:
            x_num = x_num.to(CFG.device)
            source_idx = source_idx.to(CFG.device)
            source_onehot = source_onehot.to(CFG.device)
            y = y.to(CFG.device)

            optimizer.zero_grad()

            if mode == "agnostic":
                logits = model(x_num)
            elif mode == "embedding":
                logits = model(x_num, source_idx)
            elif mode == "onehot":
                logits = model(x_num, source_onehot=source_onehot)
            else:
                raise ValueError("Unknown mode.")

            loss = criterion(logits, y)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            optimizer.step()

            total_train_loss += loss.item() * len(y)
            total_train_n += len(y)

        train_loss = total_train_loss / total_train_n
        valid_loss, _, _ = evaluate_model(model, valid_loader, mode=mode)
        scheduler.step(valid_loss)

        history.append({
            "epoch": epoch + 1,
            "train_loss": train_loss,
            "valid_loss": valid_loss,
            "lr": optimizer.param_groups[0]["lr"],
        })

        if valid_loss < best_valid_loss - 1e-5:
            best_valid_loss = valid_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        _CKPT.save_epoch_if_due(active_unit, epoch, model, optimizer, scheduler,
                                best_model_state, best_valid_loss, epochs_without_improvement, history)
        if epochs_without_improvement >= CFG.patience:
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    return model, pd.DataFrame(history)


# ============================================================
# model factory
# ============================================================

def instantiate_model(
    base_model_name: str,
    input_dim: int,
    n_sources: int,
    embedding_dim: Optional[int] = None,
):
    if base_model_name == "source_agnostic_mlp":
        return SourceAgnosticMLP(
            input_dim=input_dim,
            hidden_dims=CFG.hidden_dims,
            dropout=CFG.dropout,
        ), "agnostic"

    if base_model_name == "source_aware_embedding_mlp":
        if embedding_dim is None:
            raise ValueError("embedding_dim must be provided.")
        return SourceAwareEmbeddingMLP(
            input_dim=input_dim,
            n_sources=n_sources,
            source_embedding_dim=embedding_dim,
            hidden_dims=CFG.hidden_dims,
            dropout=CFG.dropout,
        ), "embedding"

    if base_model_name == "source_aware_onehot_mlp":
        return SourceAwareOneHotMLP(
            input_dim=input_dim,
            n_sources=n_sources,
            hidden_dims=CFG.hidden_dims,
            dropout=CFG.dropout,
        ), "onehot"

    raise ValueError("Unknown model name.")


def full_model_name(base_model_name: str, embedding_dim: Optional[int]) -> str:
    if base_model_name == "source_aware_embedding_mlp":
        return f"{base_model_name}_m{embedding_dim}"
    return base_model_name


# ============================================================
# LOSO only
# ============================================================

def run_leave_one_source_out(
    X: np.ndarray,
    y: np.ndarray,
    source_idx: np.ndarray,
    source_onehot: np.ndarray,
    source_names: np.ndarray,
    base_model_name: str,
    embedding_dim: Optional[int] = None,
):
    model_name = full_model_name(base_model_name, embedding_dim)

    results = []
    pred_rows = []
    histories = []

    unique_sources = np.unique(source_names)
    n_sources_total = source_onehot.shape[1]

    for held_out_source in unique_sources:
        unit_key = f"LOSO|{model_name}|source_{held_out_source}"
        cached = _CKPT.get_completed(unit_key)
        if cached is not None:
            print(f"[RESUME] Skipping finished {unit_key}")
            results.extend(cached['metrics'])
            pred_rows.extend(cached['predictions'])
            histories.append(pd.DataFrame(cached['history']))
            continue
        _CKPT.current_unit = unit_key
        print(f"\n[LOSO][{model_name}] held-out source: {held_out_source}")

        train_mask = source_names != held_out_source
        test_mask = source_names == held_out_source

        X_train_val = X[train_mask]
        y_train_val = y[train_mask]
        s_train_val = source_idx[train_mask]
        s1_train_val = source_onehot[train_mask]

        X_test = X[test_mask]
        y_test = y[test_mask]
        s_test = source_idx[test_mask]
        source_names_test = source_names[test_mask]

        tr_idx, va_idx = train_test_split(
            np.arange(len(y_train_val)),
            test_size=CFG.valid_size,
            stratify=y_train_val,
            random_state=CFG.random_seed,
        )

        X_train = X_train_val[tr_idx]
        y_train = y_train_val[tr_idx]
        s_train = s_train_val[tr_idx]
        s1_train = s1_train_val[tr_idx]

        X_valid = X_train_val[va_idx]
        y_valid = y_train_val[va_idx]
        s_valid = s_train_val[va_idx]
        s1_valid = s1_train_val[va_idx]

        X_train, X_valid, X_test = foldwise_median_impute_and_scale(
            X_train, X_valid, X_test
        )

        train_loader, valid_loader = make_loaders(
            X_train, s_train, s1_train, y_train,
            X_valid, s_valid, s1_valid, y_valid,
        )

        n_pos = y_train.sum()
        n_neg = len(y_train) - n_pos
        pos_weight = torch.tensor(n_neg / max(n_pos, 1.0), dtype=torch.float32)

        model, mode = instantiate_model(
            base_model_name=base_model_name,
            input_dim=X.shape[1],
            n_sources=n_sources_total,
            embedding_dim=embedding_dim,
        )

        model, history_df = train_one_model(
            model=model,
            train_loader=train_loader,
            valid_loader=valid_loader,
            mode=mode,
            pos_weight=pos_weight,
        )

        history_df["model"] = model_name
        history_df["base_model"] = base_model_name
        history_df["embedding_dim"] = embedding_dim if embedding_dim is not None else np.nan
        history_df["held_out_source"] = held_out_source
        histories.append(history_df)
        pred_start = len(pred_rows)

        if mode == "agnostic":
            test_loader = make_single_loader(
                X_test,
                np.zeros(len(y_test), dtype=int),
                np.zeros((len(y_test), n_sources_total), dtype=float),
                y_test,
            )
            _, y_true, y_prob = evaluate_model(model, test_loader, mode="agnostic")

        elif mode == "onehot":
            y_true, y_prob = evaluate_onehot_model_unseen_source(model, X_test, y_test)

        elif mode == "embedding":
            mean_embedding = model.mean_training_embedding(s_train).to(CFG.device)
            test_loader = make_single_loader(
                X_test,
                np.zeros(len(y_test), dtype=int),
                np.zeros((len(y_test), n_sources_total), dtype=float),
                y_test,
            )
            y_true, y_prob = evaluate_embedding_model_unseen_source(model, test_loader, mean_embedding)

        else:
            raise ValueError("Unknown mode.")

        metrics = compute_metrics(y_true, y_prob)
        metrics["model"] = model_name
        metrics["base_model"] = base_model_name
        metrics["embedding_dim"] = embedding_dim if embedding_dim is not None else np.nan
        metrics["held_out_source"] = held_out_source
        metrics["n_test"] = len(y_true)
        results.append(metrics)

        for i in range(len(y_true)):
            pred_rows.append({
                "model": model_name,
                "base_model": base_model_name,
                "embedding_dim": embedding_dim if embedding_dim is not None else np.nan,
                "held_out_source": held_out_source,
                "y_true": int(y_true[i]),
                "y_prob": float(y_prob[i]),
                "source": source_names_test[i],
            })

        _CKPT.complete(unit_key, metrics=[dict(metrics)], predictions=pred_rows[pred_start:],
                       history=history_df.to_dict('records'))
        _CKPT.current_unit = None

        print(
            f"AUC={metrics['auc']:.4f}, "
            f"ACC={metrics['accuracy']:.4f}, "
            f"BACC={metrics['balanced_accuracy']:.4f}, "
            f"F1={metrics['f1']:.4f}, "
            f"Precision={metrics['precision']:.4f}"
        )

    return (
        pd.DataFrame(results),
        pd.DataFrame(pred_rows),
        pd.concat(histories, axis=0, ignore_index=True),
    )


# ============================================================
# summaries
# ============================================================

def summarize_results(df_results: pd.DataFrame) -> pd.DataFrame:
    metric_cols = [
        "auc", "pr_auc", "accuracy", "balanced_accuracy", "f1", "precision",
        "recall", "sensitivity", "specificity", "npv", "mcc", "logloss", "brier"
    ]
    rows = []

    for model_name, sub in df_results.groupby("model"):
        row = {
            "model": model_name,
            "base_model": sub["base_model"].iloc[0],
            "embedding_dim": sub["embedding_dim"].iloc[0],
        }
        for col in metric_cols:
            row[f"{col}_mean"] = sub[col].mean()
            row[f"{col}_sd"] = sub[col].std(ddof=1) if len(sub) > 1 else 0.0
        rows.append(row)

    return pd.DataFrame(rows)


# ============================================================
# plotting
# ============================================================

def prettify_model_name(name: str) -> str:
    if name == "source_agnostic_mlp":
        return "Source-agnostic"
    if name == "source_aware_onehot_mlp":
        return "Source-aware one-hot"
    if name.startswith("source_aware_embedding_mlp_m"):
        m = name.split("_m")[-1]
        return f"Embedding m={m}"
    return name


def save_barplot_with_errorbars(
    df_results: pd.DataFrame,
    models_to_plot: List[str],
    metric: str,
    ylabel: str,
    filename: str,
    higher_is_better: bool = True,
):
    sub = df_results[df_results["model"].isin(models_to_plot)].copy()
    summary = (
        sub.groupby("model")[metric]
        .agg(["mean", "std"])
        .reset_index()
        .sort_values("mean", ascending=not higher_is_better)
    )

    x = np.arange(len(summary))
    plt.figure(figsize=(10, 5))
    plt.bar(x, summary["mean"].values)
    plt.errorbar(x, summary["mean"].values, yerr=summary["std"].values, fmt="none", capsize=5)
    plt.xticks(x, [prettify_model_name(m) for m in summary["model"]], rotation=20)
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} under leave-one-source-out")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_roc_plot(df_preds: pd.DataFrame, models_to_plot: List[str], filename: str):
    plt.figure(figsize=(7, 5))
    sub_all = df_preds[df_preds["model"].isin(models_to_plot)].copy()

    for model_name, sub in sub_all.groupby("model"):
        fpr, tpr, _ = roc_curve(sub["y_true"].values, sub["y_prob"].values)
        auc = roc_auc_score(sub["y_true"].values, sub["y_prob"].values)
        plt.plot(fpr, tpr, label=f"{prettify_model_name(model_name)} (AUC={auc:.3f})")

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("False positive rate")
    plt.ylabel("True positive rate")
    plt.title("ROC curves under leave-one-source-out")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_pr_plot(df_preds: pd.DataFrame, models_to_plot: List[str], filename: str):
    plt.figure(figsize=(7, 5))
    sub_all = df_preds[df_preds["model"].isin(models_to_plot)].copy()

    for model_name, sub in sub_all.groupby("model"):
        precision, recall, _ = precision_recall_curve(sub["y_true"].values, sub["y_prob"].values)
        ap = average_precision_score(sub["y_true"].values, sub["y_prob"].values)
        plt.plot(recall, precision, label=f"{prettify_model_name(model_name)} (AP={ap:.3f})")

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-recall curves under leave-one-source-out")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_calibration_plot(df_preds: pd.DataFrame, models_to_plot: List[str], filename: str, n_bins: int = 10):
    plt.figure(figsize=(7, 5))
    sub_all = df_preds[df_preds["model"].isin(models_to_plot)].copy()

    for model_name, sub in sub_all.groupby("model"):
        frac_pos, mean_pred = calibration_curve(
            sub["y_true"].values,
            sub["y_prob"].values,
            n_bins=n_bins,
            strategy="quantile",
        )
        plt.plot(mean_pred, frac_pos, marker="o", label=prettify_model_name(model_name))

    plt.plot([0, 1], [0, 1], linestyle="--")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed frequency")
    plt.title("Calibration under leave-one-source-out")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_embedding_dimension_curve(
    df_summary: pd.DataFrame,
    metric: str,
    ylabel: str,
    filename: str,
):
    sub = df_summary[df_summary["base_model"] == "source_aware_embedding_mlp"].copy()
    sub = sub.sort_values("embedding_dim")

    plt.figure(figsize=(7, 5))
    plt.errorbar(
        sub["embedding_dim"].values,
        sub[f"{metric}_mean"].values,
        yerr=sub[f"{metric}_sd"].values,
        marker="o",
        linewidth=2,
        capsize=4,
    )
    plt.xscale("log", base=2)
    plt.xlabel("Embedding dimension")
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} vs embedding dimension under LOSO")
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_per_source_metric_plot(
    df_results: pd.DataFrame,
    models_to_plot: List[str],
    metric: str,
    ylabel: str,
    filename: str,
    ylim: Optional[Tuple[float, float]] = None,
):
    sub = df_results[df_results["model"].isin(models_to_plot)].copy()
    pivot = sub.pivot(index="held_out_source", columns="model", values=metric)

    x = np.arange(len(pivot.index))
    width = 0.8 / max(1, len(models_to_plot))

    plt.figure(figsize=(10, 5))
    for j, model_name in enumerate(models_to_plot):
        if model_name not in pivot.columns:
            continue
        plt.bar(x + j * width, pivot[model_name].values, width=width, label=prettify_model_name(model_name))

    plt.xticks(x + width * (len(models_to_plot) - 1) / 2, pivot.index, rotation=20)
    plt.ylabel(ylabel)
    plt.title(f"{ylabel} by held-out source")

    if ylim is not None:
        plt.ylim(ylim)

    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def save_auc_gain_per_source_plot(
    df_results: pd.DataFrame,
    baseline_model: str,
    comparator_model: str,
    filename: str,
    ylim: Optional[Tuple[float, float]] = None,
):
    base = (
        df_results[df_results["model"] == baseline_model][["held_out_source", "auc"]]
        .rename(columns={"auc": "auc_baseline"})
    )
    comp = (
        df_results[df_results["model"] == comparator_model][["held_out_source", "auc"]]
        .rename(columns={"auc": "auc_comparator"})
    )

    merged = base.merge(comp, on="held_out_source", how="inner")
    merged["auc_gain"] = merged["auc_comparator"] - merged["auc_baseline"]
    merged = merged.sort_values("auc_gain")

    x = np.arange(len(merged))

    plt.figure(figsize=(9, 5))
    plt.bar(x, merged["auc_gain"].values)
    plt.axhline(0.0, linestyle="--", linewidth=1)
    plt.xticks(x, merged["held_out_source"].values, rotation=20)
    plt.ylabel("AUC gain")
    plt.title(
        f"AUC gain by held-out source: "
        f"{prettify_model_name(comparator_model)} minus "
        f"{prettify_model_name(baseline_model)}"
    )

    if ylim is not None:
        plt.ylim(ylim)

    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()

def save_training_curve(df_history: pd.DataFrame, model_name: str, filename: str):
    sub = df_history[df_history["model"] == model_name].copy()
    if sub.empty:
        return

    avg = sub.groupby("epoch")[["train_loss", "valid_loss"]].mean().reset_index()

    plt.figure(figsize=(7, 5))
    plt.plot(avg["epoch"], avg["train_loss"], label="Train loss")
    plt.plot(avg["epoch"], avg["valid_loss"], label="Validation loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Training curves: {prettify_model_name(model_name)}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(FIG_DIR, filename), dpi=300)
    plt.close()


def get_best_embedding_model_name(df_summary: pd.DataFrame, metric: str = "auc_mean", maximize: bool = True) -> str:
    sub = df_summary[df_summary["base_model"] == "source_aware_embedding_mlp"].copy()
    sub = sub.sort_values(metric, ascending=not maximize)
    return sub.iloc[0]["model"]


# ============================================================
# main
# ============================================================

def main():
    df, X, y, source_idx, source_onehot, feature_cols, source_encoder = load_and_prepare_data(CFG.data_path)
    source_names = source_encoder.inverse_transform(source_idx)
    _CKPT.register('LOSO', CFG, df)

    print("Dataset shape:", df.shape)
    print("Feature columns:", feature_cols)
    print("Source levels:", list(source_encoder.classes_))
    print("Embedding dimensions explored:", CFG.embedding_dims)
    print("Device:", CFG.device)

    run_specs = [("source_agnostic_mlp", None)]
    run_specs += [("source_aware_embedding_mlp", m) for m in CFG.embedding_dims]
    run_specs += [("source_aware_onehot_mlp", None)]

    all_results = []
    all_preds = []
    all_histories = []

    for base_model_name, embedding_dim in run_specs:
        res_df, pred_df, hist_df = run_leave_one_source_out(
            X=X,
            y=y,
            source_idx=source_idx,
            source_onehot=source_onehot,
            source_names=source_names,
            base_model_name=base_model_name,
            embedding_dim=embedding_dim,
        )
        all_results.append(res_df)
        all_preds.append(pred_df)
        all_histories.append(hist_df)

    df_results = pd.concat(all_results, axis=0, ignore_index=True)
    df_preds = pd.concat(all_preds, axis=0, ignore_index=True)
    df_history = pd.concat(all_histories, axis=0, ignore_index=True)

    summary = summarize_results(df_results)
    best_embedding_model = get_best_embedding_model_name(summary, metric="auc_mean", maximize=True)

    print("\nBest embedding model by mean AUC:", best_embedding_model)
    print("\nSummary:")
    print(summary.sort_values("auc_mean", ascending=False))

    df_results.to_csv(os.path.join(TABLE_DIR, "loso_results.csv"), index=False)
    df_preds.to_csv(os.path.join(TABLE_DIR, "loso_predictions.csv"), index=False)
    df_history.to_csv(os.path.join(TABLE_DIR, "loso_training_histories.csv"), index=False)
    summary.to_csv(os.path.join(TABLE_DIR, "loso_summary.csv"), index=False)

    core_models = ["source_agnostic_mlp", best_embedding_model]

    save_barplot_with_errorbars(df_results, core_models, "auc", "AUC", "figure_loso_auc.png", True)
    save_barplot_with_errorbars(df_results, core_models, "balanced_accuracy", "Balanced accuracy", "figure_loso_bacc.png", True)
    save_barplot_with_errorbars(df_results, core_models, "f1", "F1 score", "figure_loso_f1.png", True)
    save_barplot_with_errorbars(df_results, core_models, "precision", "Precision", "figure_loso_precision.png", True)
    save_barplot_with_errorbars(df_results, core_models, "specificity", "Specificity", "figure_loso_specificity.png", True)
    save_barplot_with_errorbars(df_results, core_models, "logloss", "Log-loss", "figure_loso_logloss.png", False)
    save_barplot_with_errorbars(df_results, core_models, "brier", "Brier score", "figure_loso_brier.png", False)

    save_roc_plot(df_preds, core_models, "figure_loso_roc.png")
    save_pr_plot(df_preds, core_models, "figure_loso_pr_curve.png")
    save_calibration_plot(df_preds, core_models, "figure_loso_calibration.png", n_bins=10)

    save_embedding_dimension_curve(summary, "auc", "AUC", "figure_loso_embedding_dim_auc.png")
    save_embedding_dimension_curve(summary, "balanced_accuracy", "Balanced accuracy", "figure_loso_embedding_dim_bacc.png")
    save_embedding_dimension_curve(summary, "logloss", "Log-loss", "figure_loso_embedding_dim_logloss.png")

    save_per_source_metric_plot(
        df_results, core_models, "auc", "AUC",
        "figure_loso_auc_by_source.png",
        ylim=None,
    )
    save_per_source_metric_plot(df_results, core_models, "balanced_accuracy", "Balanced accuracy",
        "figure_loso_bacc_by_source.png",)
    save_per_source_metric_plot(df_results, core_models, "logloss", "Log-loss",
        "figure_loso_logloss_by_source.png",)
    save_auc_gain_per_source_plot(
        df_results=df_results,
        baseline_model="source_agnostic_mlp",
        comparator_model=best_embedding_model,
        filename="figure_loso_auc_gain_by_source.png",
        ylim=None,
    )

    save_training_curve(df_history, best_embedding_model, "figure_loso_training_curve_best_embedding.png")

    print("\nSaved tables in:")
    print(TABLE_DIR)
    print("\nSaved figures in:")
    print(FIG_DIR)


main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Existing Google Drive checkpoint found. [R]esume or [S]tart from scratch? [R]: S
[CHECKPOINT] RESTART: previous checkpoint archived at /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/checkpoints/heart_study_checkpoint_backup_20261028_152809_769072.pt
[CHECKPOINT] Saved to: /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/checkpoints/heart_study_checkpoint.pt
[CHECKPOINT] Interval: 600.0 seconds, plus each completed model run.
[DATA] Reusing original CSV from persistent backup.
[DATA] /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/input/heart_disease_original_input.csv: original=920, duplicates_removed=2, retained=918, predictors=12 (cholesterol excluded)
[DATA] Hospitals: {'cleveland': 303, 'hungarian': 293, 'switzerland': 123, 'va_long_beach': 199}
[DATA] Missing values BEFORE fold-wise imputation: {'trestbps': 1}
[CHECKPOIN

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[CHECKPOINT] Completed CV|source_agnostic_mlp|split_4; 4 runs stored.
AUC=0.8694, ACC=0.7826, BACC=0.7752, F1=0.8113, LogLoss=0.4437

[source_agnostic_mlp] split 5/10
[CHECKPOINT] Completed CV|source_agnostic_mlp|split_5; 5 runs stored.
AUC=0.8140, ACC=0.7065, BACC=0.7066, F1=0.7273, LogLoss=0.5692

[source_agnostic_mlp] split 6/10
[CHECKPOINT] Completed CV|source_agnostic_mlp|split_6; 6 runs stored.
AUC=0.7972, ACC=0.7391, BACC=0.7360, F1=0.7647, LogLoss=0.5649

[source_agnostic_mlp] split 7/10


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:407: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(


[CHECKPOINT] Completed CV|source_agnostic_mlp|split_7; 7 runs stored.
AUC=0.9455, ACC=0.9022, BACC=0.9046, F1=0.9091, LogLoss=0.3396

[source_agnostic_mlp] split 8/10


...

Saved: /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/heart_disease_application_embedding_sweep/figures/figure_calibration_by_source_improved_hungarian.png


/tmp/ipykernel_641/3032953122.py:1440: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/heart_disease_application_embedding_sweep/figures/figure_calibration_by_source_improved_switzerland.png


/tmp/ipykernel_641/3032953122.py:1440: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Saved: /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/heart_disease_application_embedding_sweep/figures/figure_calibration_by_source_improved_va_long_beach.png
[DATA] Reusing original CSV from persistent backup.
[DATA] /content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/input/heart_disease_original_input.csv: original=920, duplicates_removed=2, retained=918, predictors=12 (cholesterol excluded)
[DATA] Hospitals: {'cleveland': 303, 'hungarian': 293, 'switzerland': 123, 'va_long_beach': 199}
[DATA] Missing values BEFORE fold-wise imputation: {'trestbps': 1}
Saved:
/content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/heart_disease_application_embedding_sweep/tables/source_descriptives.csv
/content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/heart_disease_application_embedding_sweep/tables/source_standardized_mean_differences.csv
/content/drive/MyDrive/Heart_Disease_918_Source_Embeddings/heart_disease_application_embedding_sweep/tables/source_top_heterogeneit